# Лабораторная работа 4, Самсонов Савелий Артёмович М8О-406Б-21

### Выбор задач и датасетов

Киноиндустрия сталкивается с серьезной проблемой при прогнозировании успеха фильмов на конкурентном рынке. Понимание ключевых факторов, влияющих на это и на получение дохода от фильма, имеет решающее значение для продюсеров, студий и инвесторов при принятии стратегических решений. Поэтому для определения переменных, влияющих на успех фильма, таких как бюджет, маркетинговые расходы, продолжительность фильма и рейтинги главных актеров, режиссеров и критиков, необходимо прогностическое моделирование. Для обучения моделей, которые могут предсказать успешность фильмов в прокате, взяты датасеты Movie_classification.csv и Movie_regression.xls, опубликованные на kaggle.

### Выбор метрик для классификации

1. Accuracy, измеряет долю правильно классифицированных экземпляров от общего числа примеров. Простая и понятная метрика, но может давать ошибочное представление для несбалансированных классов (когда классы имеют существенно отличающееся количество экземпляров).
2. Precision, измеряет долю правильных положительных предсказаний среди всех предсказанных положительных примеров.
3. Recall, измеряет долю правильно предсказанных положительных примеров среди всех реальных положительных примеров.
Precision, Recall важны в случае несбалансированных классов и при необходимости минимизировать ложные срабатывания.
4. F1-мера, является гармоническим средним между точностью и полнотой и используется для сбалансирования этих двух метрик. Комбинирует точность и полноту, идеально подходит для несбалансированных задач.

### Выбор метрик для регрессии

1. R², отношение между суммой квадратов отклонений предсказанных значений от среднего значения и суммой квадратов отклонений истинных значений от среднего. Показывает, какая доля вариации в целевой переменной объясняется моделью. Хороший показатель R² близкий к 1 означает, что модель хорошо объясняет данные, однако для некоторых типов задач (например, с незначительными отклонениями) значение R² может быть не таким информативным.
2. MAE, измеряет среднее абсолютное отклонение между предсказанными и истинными значениями. Полезна, когда важно понять, насколько в среднем модель ошибается по величине предсказанных значений. MAE не так чувствительна к выбросам, как другие метрики.
3. MSE, измеряет средний квадрат разницы между предсказанными и истинными значениями. MSE часто используется, когда важно акцентировать внимание на больших ошибках. Более чувствительна к выбросам, чем MAE, и может быть полезна, когда крупные ошибки особенно нежелательны.

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

In [2]:
import os
dataset_path = '.\\input'

for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

.\input\Movie_classification.csv
.\input\Movie_regression.xls


## 2. Создание бейзлайна и оценка качества

### Обучение модели из sklearn (для классификации) и оценка качества по выбранным метрикам

Загрузим датасет

In [34]:
df = pd.read_csv(dataset_path + "\\Movie_classification.csv")
df

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,3D_available,Time_taken,Twitter_hastags,Genre,Avg_age_actors,Num_multiplex,Collection,Start_Tech_Oscar
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,YES,109.60,223.840,Thriller,23,494,48000,1
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,NO,146.64,243.456,Drama,42,462,43200,0
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,NO,147.88,2022.400,Comedy,38,458,69400,1
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,YES,185.36,225.344,Drama,45,472,66800,1
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,NO,176.48,225.792,Drama,55,395,72400,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,492480,NO,186.96,243.584,Action,27,561,44800,0
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,482875,YES,132.24,263.296,Action,20,600,41200,0
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,532239,NO,109.56,243.824,Comedy,31,576,47800,0
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,496077,YES,158.80,303.520,Comedy,47,607,44000,0


Удалим некоторые параметры

In [35]:
if "Genre" in df:
  del df["Genre"]
if "3D_available" in df:
  del df["3D_available"]
if "Time_taken" in df:
  del df["Time_taken"]

Просмотрим информацию о значениях полей и убедимся, что все они допустимы

In [37]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Marketing expense    506 non-null    float64
 1   Production expense   506 non-null    float64
 2   Multiplex coverage   506 non-null    float64
 3   Budget               506 non-null    float64
 4   Movie_length         506 non-null    float64
 5   Lead_ Actor_Rating   506 non-null    float64
 6   Lead_Actress_rating  506 non-null    float64
 7   Director_rating      506 non-null    float64
 8   Producer_rating      506 non-null    float64
 9   Critic_rating        506 non-null    float64
 10  Trailer_views        506 non-null    int64  
 11  Twitter_hastags      506 non-null    float64
 12  Avg_age_actors       506 non-null    int64  
 13  Num_multiplex        506 non-null    int64  
 14  Collection           506 non-null    int64  
 15  Start_Tech_Oscar     506 non-null    int

Создадим выборки для обучения и тестирования

In [38]:
X1 = df.drop('Start_Tech_Oscar', axis=1)
y1 = df['Start_Tech_Oscar']

X1_train,X1_test,y1_train,y1_test = train_test_split(X1.values, y1.values, random_state = 0)

Обучение модели для классификации

In [41]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

sk_forest_clf = RandomForestClassifier()
sk_forest_clf.fit(X1_train, y1_train)
sk_forest_clf_pred_res = sk_forest_clf.predict(X1_test)
sk_forest_clf_accuracy = accuracy_score(y1_test, sk_forest_clf_pred_res)
sk_forest_clf_precision = precision_score(y1_test, sk_forest_clf_pred_res)
sk_forest_clf_recall = recall_score(y1_test, sk_forest_clf_pred_res)
sk_forest_clf_f1 = f1_score(y1_test, sk_forest_clf_pred_res)

print(f'sk forest classifier accuracy: {sk_forest_clf_accuracy:}')
print(f'sk forest classifier precision: {sk_forest_clf_precision:}')
print(f'sk forest classifier recall: {sk_forest_clf_recall:}')
print(f'sk forest classifier f1: {sk_forest_clf_f1:}')

sk forest classifier accuracy: 0.5748031496062992
sk forest classifier precision: 0.6527777777777778
sk forest classifier recall: 0.618421052631579
sk forest classifier f1: 0.6351351351351351


### Обучение модели из sklearn (для регрессии) и оценка качества по выбранным метрикам

Загрузим датасет

In [42]:
df2 = pd.read_csv(dataset_path + "\\Movie_regression.xls")
df2

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,3D_available,Time_taken,Twitter_hastags,Genre,Avg_age_actors,Num_multiplex,Collection
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,YES,109.60,223.840,Thriller,23,494,48000
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,NO,146.64,243.456,Drama,42,462,43200
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,NO,147.88,2022.400,Comedy,38,458,69400
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,YES,185.36,225.344,Drama,45,472,66800
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,NO,176.48,225.792,Drama,55,395,72400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
501,21.2526,78.86,0.427,36624.115,142.6,8.680,8.775,8.620,8.970,6.80,492480,NO,186.96,243.584,Action,27,561,44800
502,20.9054,78.86,0.427,33996.600,150.2,8.780,8.945,8.770,8.930,7.80,482875,YES,132.24,263.296,Action,20,600,41200
503,21.2152,78.86,0.427,38751.680,164.5,8.830,8.970,8.855,9.010,7.80,532239,NO,109.56,243.824,Comedy,31,576,47800
504,22.1918,78.86,0.427,37740.670,162.8,8.730,8.845,8.800,8.845,6.80,496077,YES,158.80,303.520,Comedy,47,607,44000


Просмотрим информацию о значениях полей для проверки, что все они допустимы

In [43]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Marketing expense    506 non-null    float64
 1   Production expense   506 non-null    float64
 2   Multiplex coverage   506 non-null    float64
 3   Budget               506 non-null    float64
 4   Movie_length         506 non-null    float64
 5   Lead_ Actor_Rating   506 non-null    float64
 6   Lead_Actress_rating  506 non-null    float64
 7   Director_rating      506 non-null    float64
 8   Producer_rating      506 non-null    float64
 9   Critic_rating        506 non-null    float64
 10  Trailer_views        506 non-null    int64  
 11  3D_available         506 non-null    object 
 12  Time_taken           494 non-null    float64
 13  Twitter_hastags      506 non-null    float64
 14  Genre                506 non-null    object 
 15  Avg_age_actors       506 non-null    int

Удалим некоторые параметры

In [44]:
del df2['3D_available']
del df2['Genre']
del df2['Time_taken']
df2.head()

,Marketing expense,Production expense,Multiplex coverage,Budget,Movie_length,Lead_ Actor_Rating,Lead_Actress_rating,Director_rating,Producer_rating,Critic_rating,Trailer_views,Twitter_hastags,Avg_age_actors,Num_multiplex,Collection
0,20.1264,59.62,0.462,36524.125,138.7,7.825,8.095,7.910,7.995,7.94,527367,223.840,23,494,48000
1,20.5462,69.14,0.531,35668.655,152.4,7.505,7.650,7.440,7.470,7.44,494055,243.456,42,462,43200
2,20.5458,69.14,0.531,39912.675,134.6,7.485,7.570,7.495,7.515,7.44,547051,2022.400,38,458,69400
3,20.6474,59.36,0.542,38873.890,119.3,6.895,7.035,6.920,7.020,8.26,516279,225.344,45,472,66800
4,21.3810,59.36,0.542,39701.585,127.7,6.920,7.070,6.815,7.070,8.26,531448,225.792,55,395,72400


Создадим выборки для обучения и тестирования

In [45]:
X2 = df2.drop('Collection', axis=1)
y2 = df2['Collection']

X2_train,X2_test,y2_train,y2_test = train_test_split(X2.values, y2.values, random_state = 42)

Обучение модели для регрессии

In [79]:
sk_forest_reg = RandomForestRegressor()
sk_forest_reg.fit(X2_train, y2_train)
sk_forest_reg_pred_res = sk_forest_reg.predict(X2_test)
sk_forest_reg_r2 = r2_score(y2_test, sk_forest_reg_pred_res)
sk_forest_reg_mae = mean_absolute_error(y2_test, sk_forest_reg_pred_res)
sk_forest_reg_mse = mean_squared_error(y2_test, sk_forest_reg_pred_res)

print(f'sk forest regressor r2: {sk_forest_reg_r2}')
print(f'sk forest regressor mae: {sk_forest_reg_mae}')
print(f'sk forest regressor mse: {sk_forest_reg_mse}')

sk forest regressor r2: 0.8349172600361172
sk forest regressor mae: 4542.283464566929
sk forest regressor mse: 46522987.496062994


## 3. Улучшение бейзлайна

### Сформулировать гипотезы (препроцессинг данных, визуализация данных, формирование новых признаков, подбор гиперпараметров на кросс-валидации и т.д.)

1. Формирование новых признаков: для задачи классификации можно создать новые признаки, комбинирующие в себе расходы и рейтинг участвующих в создании фильма людей соответственно.
2. Масштабирование данных во время предобработки
3. Подбор гиперпараметров
    - Количество деревьев n_estimators
    - Максимальная глубина деревьев max_depth
    - Минимальное количество образцов для разделения узла min_samples_split
    - Минимальное количество образцов в листе min_samples_leaf
    - Критерий разбиения criterion (gini или entropy для классификации squared_error или absolute_error для регрессии)
    - Максимальное количество признаков для построении каждого дерева max_features

#### Задача классификации

1. Формирование новых признаков

In [48]:
data3 = df.copy()
data3["Expense"] = data3["Marketing expense"] + data3["Production expense"]
data3["Rating"] = data3["Lead_ Actor_Rating"] + data3["Lead_Actress_rating"] + data3["Director_rating"] + data3["Producer_rating"]

In [49]:
data4 = data3.copy()
data4 = data4.drop(["Marketing expense","Production expense","Lead_ Actor_Rating","Lead_Actress_rating","Director_rating","Producer_rating"],axis = 1)

In [50]:
X1_new = data4.drop('Start_Tech_Oscar', axis=1)
y1_new = data4['Start_Tech_Oscar']

X1_train_new,X1_test_new,y1_train_new,y1_test_new = train_test_split(X1_new.values, y1_new.values, random_state = 0)

Проверим, как повлияло введение новых признаков

In [53]:
sk_forest_clf = RandomForestClassifier()
sk_forest_clf.fit(X1_train_new, y1_train_new)
sk_forest_clf_pred_res = sk_forest_clf.predict(X1_test_new)
sk_forest_clf_accuracy = accuracy_score(y1_test_new, sk_forest_clf_pred_res)
sk_forest_clf_precision = precision_score(y1_test_new, sk_forest_clf_pred_res)
sk_forest_clf_recall = recall_score(y1_test_new, sk_forest_clf_pred_res)
sk_forest_clf_f1 = f1_score(y1_test_new, sk_forest_clf_pred_res)

print(f'sk forest classifier accuracy: {sk_forest_clf_accuracy:}')
print(f'sk forest classifier precision: {sk_forest_clf_precision:}')
print(f'sk forest classifier recall: {sk_forest_clf_recall:}')
print(f'sk forest classifier f1: {sk_forest_clf_f1:}')

sk forest classifier accuracy: 0.6062992125984252
sk forest classifier precision: 0.696969696969697
sk forest classifier recall: 0.6052631578947368
sk forest classifier f1: 0.6478873239436619


2. Масштабирование данных

In [56]:
# scaler = StandardScaler()
# X1_train_scaled = scaler.fit_transform(X1_train)
# X1_test_scaled = scaler.transform(X1_test)

scaler = StandardScaler()
X1_train_new_scaled = scaler.fit_transform(X1_train_new)
X1_test_new_scaled = scaler.transform(X1_test_new)

Попробуем добавить к формированию новых признаков масштабирование данных

In [64]:
sk_forest_clf = RandomForestClassifier()
sk_forest_clf.fit(X1_train_new_scaled, y1_train)
sk_forest_clf_pred_res = sk_forest_clf.predict(X1_test_new_scaled)
sk_forest_clf_accuracy = accuracy_score(y1_test_new, sk_forest_clf_pred_res)
sk_forest_clf_precision = precision_score(y1_test_new, sk_forest_clf_pred_res)
sk_forest_clf_recall = recall_score(y1_test_new, sk_forest_clf_pred_res)
sk_forest_clf_f1 = f1_score(y1_test_new, sk_forest_clf_pred_res)

print(f'sk forest classifier accuracy: {sk_forest_clf_accuracy:}')
print(f'sk forest classifier precision: {sk_forest_clf_precision:}')
print(f'sk forest classifier recall: {sk_forest_clf_recall:}')
print(f'sk forest classifier f1: {sk_forest_clf_f1:}')

sk forest classifier accuracy: 0.6062992125984252
sk forest classifier precision: 0.6805555555555556
sk forest classifier recall: 0.6447368421052632
sk forest classifier f1: 0.6621621621621623


3. Попробуем добавить к формированию новых признаков и масштабированию данных Подбор гиперпараметров

In [84]:
from sklearn.model_selection import GridSearchCV

param_grid_class = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 4, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

forest_class = RandomForestClassifier()
grid_search_class = GridSearchCV(forest_class, param_grid_class, cv=5, scoring='accuracy')
grid_search_class.fit(X1_train_new_scaled, y1_train_new)
best_forest_class = grid_search_class.best_estimator_

print("Лучшие параметры для классификации:", grid_search_class.best_params_)

y_pred_class = best_forest_class.predict(X1_test_new_scaled)
accuracy = accuracy_score(y1_test_new, y_pred_class)
precision = precision_score(y1_test_new, y_pred_class)
recall = recall_score(y1_test_new, y_pred_class)
f1 = f1_score(y1_test_new, y_pred_class)

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")

Лучшие параметры для классификации (HAR): {'max_depth': None, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Accuracy: 0.5984251968503937
Precision: 0.6811594202898551
Recall: 0.618421052631579
F1 Score: 0.6482758620689655


#### Задача регрессии

2. Масштабирование данных

In [66]:
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

In [68]:
sk_forest_reg = RandomForestRegressor()
sk_forest_reg.fit(X2_train_scaled, y2_train)
sk_forest_reg_pred_res = sk_forest_reg.predict(X2_test_scaled)
sk_forest_reg_r2 = r2_score(y2_test, sk_forest_reg_pred_res)
sk_forest_reg_mae = mean_absolute_error(y2_test, sk_forest_reg_pred_res)
sk_forest_reg_mse = mean_squared_error(y2_test, sk_forest_reg_pred_res)

print(f'sk forest regressor r2: {sk_forest_reg_r2}')
print(f'sk forest regressor mae: {sk_forest_reg_mae}')
print(f'sk forest regressor mse: {sk_forest_reg_mse}')

sk forest regressor r2: 0.8415663511805703
sk forest regressor mae: 4506.44094488189
sk forest regressor mse: 44649166.01574803


3. Подбор гиперпараметров

In [69]:
param_grid_reg = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 4, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

forest_reg = RandomForestRegressor()
grid_search_reg = GridSearchCV(forest_reg, param_grid_reg, cv=5, scoring='r2')
grid_search_reg.fit(X2_train, y2_train)
best_forest_reg = grid_search_reg.best_estimator_

print("Лучшие параметры для регрессии:", grid_search_reg.best_params_)

y_pred_reg = best_forest_reg.predict(X2_test)
r2 = r2_score(y2_test, y_pred_reg)
mae = mean_absolute_error(y2_test, y_pred_reg)
mse = mean_squared_error(y2_test, y_pred_reg)

print(f"r2: {r2}")
print("mae:", mae)
print("mse:", mse)

Лучшие параметры для регрессии (CO2): {'max_depth': 30, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
r2: 0.8504276610100046
mae: 4176.850393700787
mse: 42151905.511811025


### Выводы

Введенные улучшения - формирование новых признаков и масштабирование для задачи классификации, масштабирование и подбор параметров для задачи регрессии - позволили несколько улучшить результат.

## 4. Имплементация алгоритма машинного обучения 

### Самостоятельная имплементация алгоритмов машинного обучения для классификации и регрессии

Базовый класс дерева

In [71]:
from collections import Counter

class DecisionTreeCustom:
    def __init__(self, max_depth=None, min_samples_split=2, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.tree = None

    def _gini(self, y):
        if len(y) == 0:
            return 0
        p = np.bincount(y) / len(y)
        return 1 - np.sum(p ** 2)

    def _gini_impurity(self, left_y, right_y):
        total = len(left_y) + len(right_y)
        return (len(left_y) / total) * self._gini(left_y) + (len(right_y) / total) * self._gini(right_y)

    def _leaf_value(self, y):
        return Counter(y).most_common(1)[0][0]

    def _best_split(self, X, y):
        n_samples, n_features = X.shape

        if isinstance(self.max_features, str):
            if self.max_features == "sqrt":
                n_selected_features = int(np.sqrt(n_features))
            elif self.max_features == "log2":
                n_selected_features = int(np.log2(n_features))
            else:
                raise ValueError(f"Неизвестное значение max_features: {self.max_features}")
        elif self.max_features is None:
            n_selected_features = n_features
        else:
            n_selected_features = self.max_features

        features = np.random.choice(n_features, n_selected_features, replace=False)

        best_gini = float("inf")
        best_split = None

        for feature in features:
            thresholds = np.unique(X[:, feature])
            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                left_y, right_y = y[left_mask], y[right_mask]

                if len(left_y) >= self.min_samples_split and len(right_y) >= self.min_samples_split:
                    gini = self._gini_impurity(left_y, right_y)
                    if gini < best_gini:
                        best_gini = gini
                        best_split = (feature, threshold, left_mask, right_mask)

        return best_split

    def _build_tree(self, X, y, depth=0):
        if depth == self.max_depth or len(y) < self.min_samples_split or len(set(y)) == 1:
            return self._leaf_value(y)

        split = self._best_split(X, y)
        if split is None:
            return self._leaf_value(y)

        feature, threshold, left_mask, right_mask = split
        left_tree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_tree = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return {"feature": feature, "threshold": threshold, "left": left_tree, "right": right_tree}

    def fit(self, X, y):
        self.tree = self._build_tree(X, y)

    def _predict_sample(self, x, tree):
        if not isinstance(tree, dict):
            return tree

        feature_value = x[tree["feature"]]
        if feature_value <= tree["threshold"]:
            return self._predict_sample(x, tree["left"])
        else:
            return self._predict_sample(x, tree["right"])

    def predict(self, X):
        return np.array([self._predict_sample(row, self.tree) for row in X])

Реализация для классификации

In [72]:
class CustomRandomForestClassifier:
    def __init__(self, n_estimators=10, max_depth=None, min_samples_split=2, max_features=None, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)

        self.trees = []
        for _ in range(self.n_estimators):
            n_samples = X.shape[0]
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_sample, y_sample = X[indices], y[indices]
        
            tree = DecisionTreeCustom(max_depth=self.max_depth, min_samples_split=self.min_samples_split, max_features=self.max_features)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        majority_votes = np.apply_along_axis(lambda x: Counter(x).most_common(1)[0][0], axis=0, arr=tree_preds)
        return majority_votes

Реализация для регрессии

In [73]:
class CustomRandomForestRegressor:
    def __init__(self, n_estimators=10, max_depth=None, min_samples_split=2, max_features=None, random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        if self.random_state:
            np.random.seed(self.random_state)

        self.trees = []
        for _ in range(self.n_estimators):
            n_samples = X.shape[0]
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_sample, y_sample = X[indices], y[indices]
    
            tree = DecisionTreeCustom(max_depth=self.max_depth, min_samples_split=self.min_samples_split, max_features=self.max_features)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        return np.mean(tree_preds, axis=0)


### Обучение имплементированных моделей

In [77]:
custom_forest_clf = CustomRandomForestClassifier(n_estimators=50, max_depth=10, min_samples_split=5, max_features="sqrt", random_state=42)
custom_forest_clf.fit(X1_train, y1_train)
custom_forest_clf_pred_res = custom_forest_clf.predict(X1_test)
custom_forest_clf_accuracy = accuracy_score(y1_test, custom_forest_clf_pred_res)
custom_forest_clf_precision = precision_score(y1_test, custom_forest_clf_pred_res)
custom_forest_clf_recall = recall_score(y1_test, custom_forest_clf_pred_res)
custom_forest_clf_f1 = f1_score(y1_test, custom_forest_clf_pred_res)

print(f'custom forest classifier accuracy: {custom_forest_clf_accuracy:}')
print(f'custom forest classifier precision: {custom_forest_clf_precision:}')
print(f'custom forest classifier recall: {custom_forest_clf_recall:}')
print(f'custom forest classifier f1: {custom_forest_clf_f1:}')

custom forest classifier accuracy: 0.5748031496062992
custom forest classifier precision: 0.6774193548387096
custom forest classifier recall: 0.5526315789473685
custom forest classifier f1: 0.6086956521739131


In [78]:
custom_forest_reg = CustomRandomForestRegressor(n_estimators=50, max_depth=10, min_samples_split=5, max_features=5, random_state=42)
custom_forest_reg.fit(X2_train, y2_train)
custom_forest_reg_pred_res = custom_forest_reg.predict(X2_test)
custom_forest_reg_r2 = r2_score(y2_test, custom_forest_reg_pred_res)
custom_forest_reg_mae = mean_absolute_error(y2_test, custom_forest_reg_pred_res)
custom_forest_reg_mse = mean_squared_error(y2_test, custom_forest_reg_pred_res)

print(f'custom forest regressor r2: {custom_forest_reg_r2}')
print(f'custom forest regressor mae: {custom_forest_reg_mae}')
print(f'custom forest regressor mse: {custom_forest_reg_mse}')

custom forest regressor r2: 0.7319929797220635
custom forest regressor mae: 6225.7322834645665
custom forest regressor mse: 75528715.21259843


### Выводы

Полученные результаты приемлемы, но несколько уступают результатам встроенных моделей. Возможно, применение оптимизаций улучшит ситуацию.

### Обучение имплементированных моделей в улучшенном бейзлайне

#### Задача классификации

Проверим, как повлияет введение новых признаков

In [81]:
custom_forest_clf = CustomRandomForestClassifier(n_estimators=50, max_depth=10, min_samples_split=5, max_features="sqrt", random_state=42)
custom_forest_clf.fit(X1_train_new, y1_train_new)
custom_forest_clf_pred_res = custom_forest_clf.predict(X1_test_new)
custom_forest_clf_accuracy = accuracy_score(y1_test_new, custom_forest_clf_pred_res)
custom_forest_clf_precision = precision_score(y1_test_new, custom_forest_clf_pred_res)
custom_forest_clf_recall = recall_score(y1_test_new, custom_forest_clf_pred_res)
custom_forest_clf_f1 = f1_score(y1_test_new, custom_forest_clf_pred_res)

print(f'custom forest classifier accuracy: {custom_forest_clf_accuracy:}')
print(f'custom forest classifier precision: {custom_forest_clf_precision:}')
print(f'custom forest classifier recall: {custom_forest_clf_recall:}')
print(f'custom forest classifier f1: {custom_forest_clf_f1:}')

custom forest classifier accuracy: 0.5511811023622047
custom forest classifier precision: 0.6376811594202898
custom forest classifier recall: 0.5789473684210527
custom forest classifier f1: 0.6068965517241379


Проверим, как повлияет масштабирование

In [82]:
scaler = StandardScaler()
X1_train_scaled = scaler.fit_transform(X1_train)
X1_test_scaled = scaler.transform(X1_test)

custom_forest_clf = CustomRandomForestClassifier(n_estimators=50, max_depth=10, min_samples_split=5, max_features="sqrt", random_state=42)
custom_forest_clf.fit(X1_train_scaled, y1_train)
custom_forest_clf_pred_res = custom_forest_clf.predict(X1_test_scaled)
custom_forest_clf_accuracy = accuracy_score(y1_test, custom_forest_clf_pred_res)
custom_forest_clf_precision = precision_score(y1_test, custom_forest_clf_pred_res)
custom_forest_clf_recall = recall_score(y1_test, custom_forest_clf_pred_res)
custom_forest_clf_f1 = f1_score(y1_test, custom_forest_clf_pred_res)

print(f'custom forest classifier accuracy: {custom_forest_clf_accuracy:}')
print(f'custom forest classifier precision: {custom_forest_clf_precision:}')
print(f'custom forest classifier recall: {custom_forest_clf_recall:}')
print(f'custom forest classifier f1: {custom_forest_clf_f1:}')

custom forest classifier accuracy: 0.5748031496062992
custom forest classifier precision: 0.6774193548387096
custom forest classifier recall: 0.5526315789473685
custom forest classifier f1: 0.6086956521739131


Проверим, как повлияет подбор параметров

In [85]:
from itertools import product

best_params_classification = {}
best_metrics_classification = {"accuracy": 0}

n_estimators_variants = [50, 100, 200]
max_depth_variants = [None, 4, 10, 20, 30]
min_samples_split_variants = [2, 5, 10]
max_features_variants = ['sqrt', 'log2']

for n_estimators, max_depth, min_samples_split, max_features in product(n_estimators_variants, max_depth_variants, min_samples_split_variants, max_features_variants):
    custom_tree_clf = CustomRandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split, max_features=max_features, random_state=42)
    custom_tree_clf.fit(X1_train, y1_train)
    pred_res = custom_tree_clf.predict(X1_test)

    accuracy = accuracy_score(y1_test, pred_res)
    precision = precision_score(y1_test, pred_res)
    recall = recall_score(y1_test, pred_res)
    f1 = f1_score(y1_test, pred_res)

    if accuracy > best_metrics_classification["accuracy"]:
        best_params_classification = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_split": min_samples_split,
            "max_features": max_features
        }
        best_metrics_classification = {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1
        }
        
print("Лучшие параметры для классификации:", best_params_classification)
print("Метрики для классификации:", best_metrics_classification)

Лучшие параметры для классификации: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'max_features': 'sqrt'}
Метрики для классификации: {'accuracy': 0.6377952755905512, 'precision': 0.7083333333333334, 'recall': 0.6710526315789473, 'f1': 0.6891891891891891}


#### Задача регрессии

Проверим, как повлияет масштабирование

In [87]:
custom_forest_reg = CustomRandomForestRegressor(n_estimators=50, max_depth=10, min_samples_split=5, random_state=42)
custom_forest_reg.fit(X2_train_scaled, y2_train)
custom_forest_reg_pred_res = custom_forest_reg.predict(X2_test_scaled)
custom_forest_reg_r2 = r2_score(y2_test, custom_forest_reg_pred_res)
custom_forest_reg_mae = mean_absolute_error(y2_test, custom_forest_reg_pred_res)
custom_forest_reg_mse = mean_squared_error(y2_test, custom_forest_reg_pred_res)

print(f'custom forest regressor r2: {custom_forest_reg_r2}')
print(f'custom forest regressor mae: {custom_forest_reg_mae}')
print(f'custom forest regressor mse: {custom_forest_reg_mse}')

custom forest regressor r2: 0.6946742243209043
custom forest regressor mae: 6693.417322834645
custom forest regressor mse: 86045744.37795275


Проверим, как повлияет подбор параметров

In [88]:
from itertools import product

best_params_classification = {}
best_metrics_classification = {"r2": 0}

n_estimators_variants = [50, 100, 200]
max_depth_variants = [None, 10, 20, 30]
min_samples_split_variants = [2, 5, 10]
max_features_variants = ['sqrt', 'log2']

for n_estimators, max_depth, min_samples_split, max_features in product(n_estimators_variants, max_depth_variants, min_samples_split_variants, max_features_variants):
    custom_tree_clf = CustomRandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, min_samples_split=min_samples_split, max_features=max_features, random_state=42)
    custom_tree_clf.fit(X2_train, y2_train)
    pred_res = custom_tree_clf.predict(X2_test)

    r2 = r2_score(y2_test, pred_res)
    mae = mean_absolute_error(y2_test, pred_res)
    mse = mean_squared_error(y2_test, pred_res)

    if r2 > best_metrics_classification["r2"]:
        best_params_classification = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_split": min_samples_split,
            "max_features": max_features
        }
        best_metrics_classification = {
            "r2": r2,
            "mae": mae,
            "mse": mse
        }
        
print("Лучшие параметры для классификации:", best_params_classification)
print("Метрики для классификации:", best_metrics_classification)

Лучшие параметры для классификации: {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 2, 'max_features': 'sqrt'}
Метрики для классификации: {'r2': 0.8051039012004453, 'mae': 5124.566929133858, 'mse': 54924874.45669291}


### Выводы

Как и для встроенных моделей, удалось улучшить результаты работы.
В целом, результаты собственных моделей приблизились к результатам встроенных.